# The cost of ammonia production

Analysis of the cost of ammonia based on different technology pathways.

The source of nitrogen is likely to be air separation through the course of the energy transition.

However, the hydrogen production is likely to involve carbon capture and utilization (CCU) along side tradition methane reforming production [Blue pathway]
Or electrolysis [Green pathway].

To this end, the framework is used to analyze the cost of hydrogen production as a result of the technologies avaible at our disposal, as also the trade-offs interms of emissions when considering them.


- H2 Use for Ammonia production:  400,000 to 500,000 kg H2/day; we may add other production volumes later

- H2 carbon intensity:  100% renewable electrolysis (0 kg CO2 / kg H2); 95% CCS + ATR/SMR (0.4 kg CO2/kg H2 – this qualifies for the $3/kg H2 PTC); There might be a 90% CCS case in some of the projects, but not sure yet on that. 

- H2 Cost:  The partners have provided no data, so we have to make some assumptions.  Let’s start with $1/kg H2 for now.  Or, maybe you can back out what this number should be based on the Ammonia market. 

- Ammonia price:  I’ll let you define this.  I think the goal would be for clean hydrogen based ammonia to be competitive with dirty hydrogen ammonia. 

__author__ = "Rahul Kakodkar", "Swaminathan Sundar", "Efstratios Pistikopoulos"
__copyright__ = "Copyright 2023, Multi-parametric Optimization & Control Lab"
__credits__ = ["Rahul Kakodkar", "Swaminathan Sundar", "Efstratios N. Pistikopoulos"]
__license__ = "Open"
__version__ = "1.0.0"
__maintainer__ = "Rahul Kakodkar"
__email__ = "cacodcar@tamu.edu"
__status__ = "Production"

In [1]:
import pandas 
from numpy import poly1d, polyfit, arange
from src.energiapy.components.temporal_scale import Temporal_scale
from src.energiapy.components.resource import Resource
from src.energiapy.components.process import Process
from src.energiapy.components.material import Material
from src.energiapy.components.location import Location
from src.energiapy.components.network import Network
from src.energiapy.components.scenario import Scenario
from src.energiapy.components.transport import Transport
from src.energiapy.components.result import Result 
from src.energiapy.model.formulate import formulate, Constraints, Objective
from src.energiapy.plot import plot
from src.energiapy.model.solve import solve
import matplotlib.pyplot as plt
from matplotlib import rc
from itertools import product
from pyomo.environ import Constraint, Set


**Constants**

In [1]:
bigM = 10**6
smallM = 10**(-2)

**Resources**

In [2]:
Charge = Resource(name='Charge', store_max=100, basis='MW', label='Battery energy', block='energystorage')

Air = Resource(name='Air', cons_max = bigM, basis = 'MW', label='Air', block='materialfeedstock')

Air_C = Resource(name='Air_C', store_max=bigM, basis='MW', label='CAES energy', block='energystorage')

H2O_E = Resource(name='H2O_E', store_max=bigM, basis='MW',
                 label='PSH energy', block='energystorage')

Uranium = Resource(name='Uranium', cons_max=(1/4.17*10**(-5))*bigM,
                   price=ur_price/(250/2), basis='kg', label='Uranium', block='energyfeedstock')

Solar = Resource(
    name='Solar', cons_max=bigM, basis='MW', label='Solar Power', block='energyfeedstock')

Wind = Resource(name='Wind', cons_max= bigM, basis='MW', label='Wind Power', block='energyfeedstock')

H2_L = Resource(name='H2_L', store_max=10**10, revenue=2,
                mile=1/(0.1180535*1.60934), basis='kg', label='Hydrogen - Geological', block='resourcestorage')

H2_C = Resource(name='H2_C', store_max= 10**10, loss=0.025/24, revenue=2, mile=1/(0.1180535*1.60934), \
    basis='kg', label='Hydrogen - Local Cryo', block='resourcestorage')


# H2 = Resource(name='H2', basis='kg', sell = True, demand = True, label='Hydrogen', block='Resource')
H2 = Resource(name='H2', basis='kg', label='Hydrogen', block='Resource')


H2O = Resource(name='H2O', cons_max=10**10,
               price=water_price/(5000*3.7854), basis='kg', label='Water', block='Resource')
            
O2 = Resource(name='O2', sell=True, loss=0.07,
              basis='kg', label='Oxygen', block='Resource')


CH4 = Resource(name='CH4', cons_max=100, price=0.113891, basis='kg', varying= True,\
    label='Natural gas', block='materialfeedstock')

CO2 = Resource(name='CO2', basis='kg',
               label='Carbon dioxide', block='Resource')


CO2_DAC = Resource(
    name='CO2_DAC', basis='kg', label='Carbon dioxide - captured', block='carbonsequestration')

CO2_AQoff = Resource(
    name='CO2_AQoff', store_max=10**6, basis='kg', label='Carbon dioxide - sequestered', block='carbonsequestration')

CO2_EOR = Resource(
    name='CO2_EOR', store_max=10**6, basis='kg', label='Carbon dioxide - EOR', block='carbonsequestration')


CH3OH = Resource(name='CH3OH', sell=True, revenue=0.5,
                 mile=1/(0.0195508*1.60934), basis='kg', label='Methanol', block='resourcedischarge')

CO2_Vent = Resource(
    name='CO2_Vent', sell=True, basis='kg', label='Carbon dioxide - Vented', block='resourcedischarge')

# Power= Resource(name= 'Power', sell= True, store_max=0,   \
#    mile= (10**3)/(0.2167432**1.60934), label= 'Renewable power generated')

Power = Resource(name='Power', basis='MW',
                 label='Renewable power generated', block='Resource')

Mile = Resource(name = 'Mile', basis = 'miles', sell = True, demand  = True, label = 'miles driven')


**Process**

In [3]:
#Tasks
Pasteurizer = Process(name = 'Pasteurizer', prod_min = 0, prod_max = 2700, conversion = {SkimMilk: -1, PastMilk: 1, CO2: 1.5, CH4:0.6, NO2:0.2}, \
    cost= {'CAPEX': 0, 'Fixed O&M': 50, 'Variable O&M': 0.4, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 1)
Vat1 = Process(name = 'Vat1', prod_min = 0, prod_max = 600, conversion = {Culture: -1, PastMilk: -7.33, Whey: 7.467, Curd: 0.867, CO2: 1, CH4:0.8, NO2:0.5}, \
    cost= {'CAPEX': 0, 'Fixed O&M': 75, 'Variable O&M': 0.45, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 8)
Vat2 = Process(name = 'Vat2', prod_min = 0, prod_max = 800, conversion = {Culture: -1, PastMilk: -7.33, Whey: 7.467, Curd: 0.867, CO2: 1, CH4:0.8, NO2:0.5},\
    cost= {'CAPEX': 0, 'Fixed O&M': 81, 'Variable O&M': 0.5, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 8)
Vat3 = Process(name = 'Vat3', prod_min = 0, prod_max = 1940, conversion = {Culture: -1, PastMilk: -7.33, Whey: 7.467, Curd: 0.867, CO2: 1, CH4:0.8, NO2:0.5},\
    cost= {'CAPEX': 0, 'Fixed O&M': 89, 'Variable O&M': 0.55, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 8)
Drainer = Process(name = 'Drainer', prod_min = 0, prod_max = 225, conversion = {Curd: -1, SolCurd: 0.9, Waste_1: 0.1},\
    cost= {'CAPEX': 0, 'Fixed O&M': 45, 'Variable O&M': 0.3, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 1)
Blender_1 = Process(name = 'Blender1', prod_min = 0, prod_max = 300, conversion = {Cream: -0.33, SolCurd: -0.67, Cheese: 1},\
    cost= {'CAPEX': 0, 'Fixed O&M': 30, 'Variable O&M': 0.2, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 2)
Blender_2 = Process(name = 'Blender2', prod_min = 0, prod_max = 300, conversion = {Cream: -0.33, SolCurd: -0.67, Cheese: 1},\
    cost= {'CAPEX': 0, 'Fixed O&M': 30, 'Variable O&M': 0.2, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 2)
Filler_1 = Process(name = 'Filler1', prod_min = 0, prod_max = 150, conversion = {Cheese: -1, StdBld1: 1},\
    cost= {'CAPEX': 0, 'Fixed O&M': 25, 'Variable O&M': 0.2, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 1)
Filler_2 = Process(name = 'Filler2', prod_min = 0, prod_max = 150, conversion = {Cheese: -1, StdBld2: 1}, \
    cost= {'CAPEX': 0, 'Fixed O&M': 25, 'Variable O&M': 0.2, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 1)
UltraFilMem = Process(name = 'UFMem', prod_min = 0, prod_max = 5300, conversion = {Whey: -1, Protein: 0.03 , Permeate: 0.97},\
    cost= {'CAPEX': 0, 'Fixed O&M': 45, 'Variable O&M': 0.3, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 8, label = 'Ultra_Filtratiion_Membrane')
RevOsmoMem = Process(name = 'ROMem', prod_min = 0, prod_max = 5300, conversion = {Permeate: -1, Lactose: 0.9 , Waste_2: 0.1},\
    cost= {'CAPEX': 0, 'Fixed O&M': 120, 'Variable O&M': 0.4, 'units': '$/kg', 'source': 'dummy'}, \
        intro_scale = 0, exit_scale= 2, label = 'Reverse_Osmosis_Membrane')


In [4]:

Milk_Silo = Process(name = 'MilkSilo', prod_min = 0, prod_max= 2*14100,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = SkimMilk, block = 'Storage')
Culture_Silo = Process(name = 'CultureSilo', prod_min = 0, prod_max= 2*5000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Culture, block = 'Storage')
Curd_Tank_1 = Process(name = 'CurdTank1', prod_min = 0, prod_max= 2*2000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Curd, block = 'Storage')
Curd_Tank_2 = Process(name = 'CurdTank2', prod_min = 0, prod_max= 2*2000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Curd, block = 'Storage')
Curd_Tank_3 = Process(name = 'CurdTank3', prod_min = 0, prod_max= 2*2000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Curd, block = 'Storage')
Waste_Tank_1 = Process(name = 'WasteTank1', prod_min = 0, prod_max= 2*2000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Waste_1, block = 'Storage')
Waste_Tank_2 = Process(name = 'WasteTank2', prod_min = 0, prod_max= 2*2000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Waste_2, block = 'Storage')
Cream_Silo = Process(name = 'CreamSilo', prod_min = 0, prod_max= 2*2800,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Cream, block = 'Storage')
Warehouse_1 = Process(name = 'Warehouse1', prod_min = 0, prod_max= 2*1000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = StdBld1, block = 'Storage')
Warehouse_2 = Process(name = 'Warehouse2', prod_min = 0, prod_max= 2*1000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = StdBld2, block = 'Storage')
Protein_Tank = Process(name = 'ProteinTank', prod_min = 0, prod_max= 2*3000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Protein, block = 'Storage')
Lactose_Tank = Process(name = 'LactoseTank', prod_min = 0, prod_max= 2*3000,\
    cost= {'CAPEX': 0, 'Fixed O&M': 15, 'Variable O&M': 0.1, 'units': '$/kg', 'source': 'dummy'}, storage = Lactose, block = 'Storage')

In [5]:
initial_dict = {'Cream_stored': 28,'Curd_stored':0 , 'Lactose_stored':0 , 'SkimMilk_stored':141, 'Waste1_stored':0,\
    'Culture_stored':50, 'StdBld1_stored':0, 'Protein_stored':0}

In [6]:
scales = Temporal_scale([1,6])# annual working hours, each discretization represents 30 mins

**Constants**

In [7]:
processes = {Pasteurizer, Vat1, Vat2, Vat3, Drainer, Blender_1, Blender_2, UltraFilMem, \
    RevOsmoMem, Milk_Silo, Culture_Silo, Curd_Tank_1, Curd_Tank_2, Curd_Tank_3, Waste_Tank_1,\
        Waste_Tank_2, Warehouse_1, Warehouse_2, Protein_Tank, Lactose_Tank, Filler_1, Filler_2}

In [8]:
# processes = {Pasteurizer, Vat1, Drainer, Blender_1, UltraFilMem, RevOsmoMem, Milk_Silo, \
#     Culture_Silo, Curd_Tank_1, Waste_Tank_1, Warehouse_1, Protein_Tank, Lactose_Tank, Cream_Silo, Filler_1}

In [9]:
factory = Location(name= 'Factory', processes= processes, scales = scales, label= 'cheese cake factory')

In [10]:
scenario = Scenario(name= 'shell', network= factory, scales= scales,\
    scheduling_scale_level = 1, demand_scale_level = 0, network_scale_level = 0, label= 'shell milp case study (HO)')

In [11]:
milp = formulate(scenario= scenario, demand= 100,\
        constraints={Constraints.cost, Constraints.inventory, Constraints.production, Constraints.resource_balance}, \
                objective= Objective.cost)

{StdBld1_stored, StdBld2_stored, Curd_stored, Culture_stored, Waste2_stored, Lactose_stored, Protein_stored, Waste1_stored, SkimMilk_stored}


In [12]:
# milp = formulate(scenario= scenario, \
#     constraints={Constraints.cost, Constraints.inventory, Constraints.production, Constraints.resource_balance}, \
#         objective= Objective.demand)

In [13]:
# def initial_inventory_rule(instance, location, resource):
#     return  instance.Inv[location, resource, 0] == initial_dict[resource]
# milp.initial_inventory_cons = Constraint(milp.locations, milp.resources_store, rule = initial_inventory_rule )

In [14]:
results = solve(scenario = scenario, instance= milp, solver= 'gurobi', \
    name=f"LCA case study", print_solversteps = True)

Gurobi Optimizer version 9.5.2 build v9.5.2rc0 (win64)
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 624 rows, 639 columns and 1765 nonzeros
Model fingerprint: 0x389a5c4c
Variable types: 586 continuous, 53 integer (53 binary)
Coefficient statistics:
  Matrix range     [3e-02, 3e+04]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+02, 1e+04]
Presolve removed 472 rows and 499 columns
Presolve time: 0.00s
Presolved: 152 rows, 140 columns, 430 nonzeros
Variable types: 140 continuous, 0 integer (0 binary)

Root relaxation: objective 1.608547e+05, 72 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0               0    160854.65165 160854.652  0.00%     -    0s

Explored 1 nodes (72 simplex iterations) in 0.02 seconds (0.00 work units)
Thread c

In [15]:
milp.X_P.pprint()

X_P : Process Binary
    Size=22, Index=X_P_index
    Key                           : Lower : Value : Upper : Fixed : Stale : Domain
       ('Factory', 'Blender1', 0) :     0 :   1.0 :     1 : False : False : Binary
       ('Factory', 'Blender2', 0) :     0 :   1.0 :     1 : False : False : Binary
    ('Factory', 'CultureSilo', 0) :     0 :   1.0 :     1 : False : False : Binary
      ('Factory', 'CurdTank1', 0) :     0 :   1.0 :     1 : False : False : Binary
      ('Factory', 'CurdTank2', 0) :     0 :   1.0 :     1 : False : False : Binary
      ('Factory', 'CurdTank3', 0) :     0 :   1.0 :     1 : False : False : Binary
        ('Factory', 'Drainer', 0) :     0 :   1.0 :     1 : False : False : Binary
        ('Factory', 'Filler1', 0) :     0 :   1.0 :     1 : False : False : Binary
        ('Factory', 'Filler2', 0) :     0 :   1.0 :     1 : False : False : Binary
    ('Factory', 'LactoseTank', 0) :     0 :   1.0 :     1 : False : False : Binary
       ('Factory', 'MilkSilo', 0) :  

In [16]:
milp.S.pprint()

S : Resource Dispensed/Sold
    Size=42, Index=S_index
    Key                           : Lower : Value              : Upper : Fixed : Stale : Domain
         ('Factory', 'CH4', 0, 0) :     0 :  892.6464180443396 :  None : False : False : NonNegativeReals
         ('Factory', 'CH4', 0, 1) :     0 :  892.6464180443419 :  None : False : False : NonNegativeReals
         ('Factory', 'CH4', 0, 2) :     0 :  892.6464180443419 :  None : False : False : NonNegativeReals
         ('Factory', 'CH4', 0, 3) :     0 :   892.646418044342 :  None : False : False : NonNegativeReals
         ('Factory', 'CH4', 0, 4) :     0 :   892.646418044342 :  None : False : False : NonNegativeReals
         ('Factory', 'CH4', 0, 5) :     0 :   892.646418044342 :  None : False : False : NonNegativeReals
         ('Factory', 'CO2', 0, 0) :     0 :  2059.887222862996 :  None : False : False : NonNegativeReals
         ('Factory', 'CO2', 0, 1) :     0 :  2059.887222863001 :  None : False : False : NonNegativeReals
 

In [17]:
milp

In [18]:
# milp.resource_purchase_constraint.pprint()


In [19]:
# milp.nameplate_production_constraint.pprint()

In [20]:
# milp.storage_facility_constraint.pprint()

In [21]:
# milp.location_production_constraint.pprint()

In [22]:
# milp.inventory_balance_constraint.pprint()


In [23]:
# milp.demand_constraint.pprint()

In [24]:
# milp.resources_demand.pprint()


In [25]:
# milp.resources_sell.pprint()

In [26]:
# milp.resources_purch.pprint()

In [27]:
# milp.resources_demand.pprint()

In [28]:
factory.__dict__.keys()

dict_keys(['name', 'processes', 'scales', 'demand', 'demand_factor', 'cost_factor', 'capacity_factor', 'demand_level', 'cost_level', 'capacity_level', 'label', 'resources', 'materials', 'scale_levels', 'resource_price', 'failure_processes', 'fail_factor'])

In [29]:
scenario.conversion.keys()

dict_keys(['MilkSilo', 'LactoseTank', 'ProteinTank', 'Vat1', 'Blender2', 'ROMem', 'WasteTank1', 'Warehouse1', 'Filler1', 'Warehouse2', 'Vat3', 'Drainer', 'Vat2', 'CultureSilo', 'CurdTank1', 'CurdTank3', 'WasteTank2', 'Pasteurizer', 'UFMem', 'Filler2', 'Blender1', 'CurdTank2', 'MilkSilo_discharge', 'LactoseTank_discharge', 'ProteinTank_discharge', 'WasteTank1_discharge', 'Warehouse1_discharge', 'Warehouse2_discharge', 'CultureSilo_discharge', 'CurdTank1_discharge', 'CurdTank3_discharge', 'WasteTank2_discharge', 'CurdTank2_discharge'])

In [30]:
for i in list(scenario.resource_set):
    print(i.name)

StdBld1_stored
Whey
Cream
Waste2
SolCurd
CO2
Permeate
NO2
Waste1_stored
Protein_stored
Curd_stored
StdBld2
PasteurizedMilk
StdBld2_stored
SkimMilk_stored
SkimMilk
StdBld1
Culture_stored
Cheese
Curd
Waste2_stored
Lactose_stored
CH4
Waste1
Protein
Culture
Lactose


In [31]:
milp.inventory_balance_constraint[factory.name,'StdBld1_stored',0, 1].pprint()

{Member of inventory_balance_constraint} : mass balance across scheduling scale
    Size=162, Index=inventory_balance_constraint_index, Active=True
    Key                                 : Lower : Body                                                                                                                                  : Upper : Active
    ('Factory', 'StdBld1_stored', 0, 1) :   0.0 : P[Factory,Warehouse1,0,1] - P[Factory,Warehouse1_discharge,0,1] - (Inv[Factory,StdBld1_stored,0,1] - Inv[Factory,StdBld1_stored,0,0]) :   0.0 :   True


In [34]:
milp.storage_facility_constraint.pprint()

storage_facility_constraint : storage facility sizing and location
    Size=9, Index=storage_facility_constraint_index, Active=True
    Key                               : Lower : Body                                                                    : Upper : Active
     ('Factory', 'Culture_stored', 0) :  -Inf :   Cap_S[Factory,Culture_stored,0] - 10000*X_S[Factory,Culture_stored,0] :   0.0 :   True
        ('Factory', 'Curd_stored', 0) :  -Inf :          Cap_S[Factory,Curd_stored,0] - 4000*X_S[Factory,Curd_stored,0] :   0.0 :   True
     ('Factory', 'Lactose_stored', 0) :  -Inf :    Cap_S[Factory,Lactose_stored,0] - 6000*X_S[Factory,Lactose_stored,0] :   0.0 :   True
     ('Factory', 'Protein_stored', 0) :  -Inf :    Cap_S[Factory,Protein_stored,0] - 6000*X_S[Factory,Protein_stored,0] :   0.0 :   True
    ('Factory', 'SkimMilk_stored', 0) :  -Inf : Cap_S[Factory,SkimMilk_stored,0] - 28200*X_S[Factory,SkimMilk_stored,0] :   0.0 :   True
     ('Factory', 'StdBld1_stored', 0) :  -Inf 

In [35]:
# milp.storage_facility_constraint.pprint()

In [ ]:
milp.demand_constraint.pprint()

: 

: 